# Классификация психологических состояний по текстам обсуждений

Финальное решение: TF-IDF + LinearSVC, настройка сдвигов по OOF-прогнозам и два бинарных каскада. Классы: БАР, ОКР, ПРЛ, депрессия, тревожное расстройство и шизофрения.

Модель предсказывает метки датасета для обсуждений; это исследовательский проект, а не средство диагностики. Сохранённый результат исходного эксперимента: **Macro F1 0,8513**, accuracy 0,912. Подробности протокола и ограничений — в README.

Ноутбук сокращён из `final_notebook.ipynb`: сохранён алгоритм финального решения, удалены промежуточные эксперименты. Выводы вычислительных ячеек очищены; полный повторный запуск этой версии не выполнялся.

## 1. Окружение и данные

Python 3.10+ и Jupyter. Установка зависимостей в окружение ядра:

```bash
python -m pip install numpy pandas scikit-learn pymorphy3 razdel nltk joblib jupyter
python -m nltk.downloader stopwords
```

Поместите исходный `Data_Sources_csv.csv` рядом с ноутбуком либо задайте путь в `DATA_PATH`. CSV разделён `;`, обязательные колонки: `text`, `tag`, `url`, `source`. Порядок строк важен для восстановления обсуждений ВКонтакте. Версии исходного окружения не сохранены; точное совпадение метрик при повторном запуске не гарантируется.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_colwidth", 120)

# путь к исходному CSV — поправь под свой
DATA_PATH = Path("Data_Sources_csv.csv")
N_JOBS = 1  # Увеличьте при наличии свободной памяти.

# Разделитель ';'. Внутри text встречаются кавычки и экранированные ""..."" —
# это стандартное CSV-квотирование, поэтому quotechar оставляем по умолчанию (").
# engine='python' + on_bad_lines='skip' — терпимо к битым строкам:
# в файле есть строки со сбитым числом колонок (например, без url).
df = pd.read_csv(
    DATA_PATH,
    sep=";",
    encoding="utf-8",
    engine="python",
    on_bad_lines="skip",
)

# чистим названия колонок от случайных пробелов
df.columns = [c.strip() for c in df.columns]

print("Размер:", df.shape)
print("Колонки:", list(df.columns))
assert {"text", "tag", "url", "source"}.issubset(df.columns)

## 2. Восстановление обсуждений и очистка

Для b17 используется идентификатор темы из URL; для ВКонтакте — начало поста и порядок строк. Строки без распознанного обсуждения считаются отдельными объектами. Это эвристика, а не гарантия независимости исходных авторов или обсуждений.

In [ ]:
df = df.reset_index(drop=True)                 # порядок = порядок в файле (важно для ВК cumsum)
url = df["url"].fillna("").astype(str)

b17_mask  = url.str.contains("b17.ru", case=False)
b17_topic = url.str.extract(r"id=(\d+)")[0]    # достаём число после id=

vk_mask     = url.str.contains("vk.com", case=False)
vk_is_start = vk_mask & url.str.contains(r"w=wall", case=False)
vk_thread_n = vk_is_start.cumsum()             # каждый новый wall-пост => +1 к номеру треда

thread_id = pd.Series(index=df.index, dtype="object")
thread_id[b17_mask] = "b17_" + b17_topic[b17_mask].astype(str)
thread_id[vk_mask]  = "vk_"  + vk_thread_n[vk_mask].astype(str)
df["thread_id"] = thread_id

print("Всего строк:           ", len(df))
print("Уникальных тредов:     ", df["thread_id"].nunique())
print("Строк без thread_id:   ", df["thread_id"].isna().sum())
print("\nТредов по источникам:")
print("  b17:", df.loc[b17_mask, "thread_id"].nunique())
print("  ВК: ", df.loc[vk_mask,  "thread_id"].nunique())
print("\nРазмер тредов (строк в треде):")
print(df["thread_id"].value_counts().describe())

In [ ]:
reddit_mask = df["thread_id"].isna()
df.loc[reddit_mask, "thread_id"] = ["reddit_" + str(i) for i in range(reddit_mask.sum())]
print("reddit-строк -> отдельных тредов:", int(reddit_mask.sum()))
print("Строк без thread_id осталось:", df["thread_id"].isna().sum())

DROP = ["тревожное р-во/депрессия", "тревожное р-во/невроз", "паранойя"]
df = df[~df["tag"].isin(DROP)].reset_index(drop=True)
print("\nКлассов осталось:", df["tag"].nunique())
print(df["tag"].value_counts())

print("\n=== Тредов на класс ===")
print(df.groupby("tag")["thread_id"].nunique().sort_values())

In [ ]:
n0 = len(df)

df = df[df["text"].notna()].copy()
df["text"] = df["text"].astype(str)

df = df.drop_duplicates(subset="text").reset_index(drop=True)

def clean_text(t):
    t = t.lower()
    t = re.sub(r"https?://\S+|www\.\S+", " ", t)        # ссылки
    t = re.sub(r"\[id\d+\|[^\]]+\]", " ", t)             # vk-меншены [id123|Имя]
    t = re.sub(r"\b(club|id|wall)[-\d_]+\b", " ", t)     # vk-идентификаторы
    t = re.sub(r"[^а-яёa-z0-9\s]", " ", t)               # оставляем буквы/цифры
    t = re.sub(r"\s+", " ", t).strip()
    return t

df["text_clean"] = df["text"].map(clean_text)

word_cnt = df["text_clean"].str.split().str.len()
df = df[word_cnt >= 3].reset_index(drop=True)

print(f"Строк: {n0} -> {len(df)}  (удалено {n0 - len(df)})")
print("\nПо классам после чистки:")
print(df["tag"].value_counts())

In [ ]:
# Лемматизация и кэш слов
import time
from pymorphy3 import MorphAnalyzer
from razdel import tokenize

morph = MorphAnalyzer()
_cache = {}                      # кэш слово->лемма: словарь меньше, чем все токены, => быстро
def _lemma(w):
    if w not in _cache:
        _cache[w] = morph.parse(w)[0].normal_form
    return _cache[w]

def lemmatize(t):
    return " ".join(_lemma(tok.text) for tok in tokenize(t))

t0 = time.time()
df["text_lemma"] = df["text_clean"].map(lemmatize)
print(f"Лемматизация заняла {time.time()-t0:.0f} c | уникальных слов в кэше: {len(_cache)}")



## 3. Разделение по обсуждениям

80/20 по идентификаторам обсуждений, с учётом классов и seed 42. В историческом запуске: 60 406 сообщений после очистки, 44 933 в train и 15 473 в test. Два обсуждения имели несколько меток: для стратификации исходный код берёт первую метку, для обучения на целом обсуждении — моду. Это поведение сохранено для сопоставимости; неоднозначная разметка требует отдельной проверки.

In [ ]:
# === Ячейка 7: честный сплит 80/20 ПО ТРЕДАМ + чекпойнт ===
from sklearn.model_selection import train_test_split

# метка у треда одна на всех — проверим это явно (важно для защиты)
multi = df.groupby("thread_id")["tag"].nunique()
print("Тредов с >1 меткой:", int((multi > 1).sum()), "(проверка неоднозначной разметки)")

# таблица тред -> класс, и стратифицированный сплит ИМЕННО ТРЕДОВ
threads = df.groupby("thread_id")["tag"].first().reset_index()
thr_train, thr_test = train_test_split(
    threads, test_size=0.2, stratify=threads["tag"], random_state=RANDOM_STATE
)
train_ids, test_ids = set(thr_train["thread_id"]), set(thr_test["thread_id"])

train_df = df[df["thread_id"].isin(train_ids)].reset_index(drop=True)
test_df  = df[df["thread_id"].isin(test_ids)].reset_index(drop=True)

# КОНТРОЛЬ УТЕЧКИ: пересечение тредов обязано быть пустым
print("Пересечение тредов train/test:", len(train_ids & test_ids), "(должно быть 0)")
print(f"Train: {len(train_df)} | Test: {len(test_df)} "
      f"({len(test_df)/len(df):.1%} в тесте)")
print("\nКлассы train:\n", train_df["tag"].value_counts())
print("\nКлассы test:\n",  test_df["tag"].value_counts())


assert not (train_ids & test_ids), "Обсуждения пересекаются между train и test"


In [ ]:
# === Ячейка 13: классификация НА УРОВНЕ ТРЕДА ===
# Объект = тред (весь его текст). Максимум честного контекста, без построчных повторов.
# Прогноз треда разносим на все его строки -> сравниваем по тому же row-level macro F1.
train_thr = set(train_df["thread_id"]); test_thr = set(test_df["thread_id"])

thread_doc = df.groupby("thread_id")["text_lemma"].apply(" ".join)
thread_tag = df.groupby("thread_id")["tag"].agg(lambda s: s.mode().iloc[0])  # мажоритарная метка
docs = pd.DataFrame({"doc": thread_doc, "tag": thread_tag}).reset_index()
tr = docs[docs.thread_id.isin(train_thr)]
te = docs[docs.thread_id.isin(test_thr)]
print("Тредов: train =", len(tr), "| test =", len(te))



## 4. Основная модель

TF-IDF по леммам и LinearSVC с балансировкой классов. Перебираются 32 конфигурации, 5 фолдов по обсуждениям. Параметры выбираются по train CV; в этой сокращённой версии test оценивается только после сборки каскадов. В исходных экспериментах test просматривался неоднократно — это ограничение исторического результата.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, classification_report, accuracy_score
from nltk.corpus import stopwords

# Отсутствие корпуса вызывает явную ошибку: запуск без стоп-слов меняет эксперимент.
RU_STOP = stopwords.words("russian")

In [ ]:
# === Ячейка 14: тюнинг (CV на train-тредах; test — один раз в конце) ===
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold

pipe = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=50000, stop_words=RU_STOP)),
    ("svm",   LinearSVC(class_weight="balanced")),
])
grid = {
    "tfidf__ngram_range":  [(1, 2), (1, 3)],
    "tfidf__min_df":       [2, 3],
    "tfidf__sublinear_tf": [True, False],
    "svm__C":              [0.5, 1, 2, 5],
}
gs = GridSearchCV(pipe, grid, scoring="f1_macro",
                  cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
                  n_jobs=N_JOBS, verbose=1)
gs.fit(tr["doc"], tr["tag"])                      # каждый тред = один объект -> CV безопасен по группам
print("Лучший CV macro F1:", round(gs.best_score_, 4))
print("Лучшие параметры:", gs.best_params_)



## 5. Настройка сдвигов по OOF-прогнозам

Для каждого класса подбирается добавка к decision function. Используются только прогнозы на train, полученные вне обучающего фолда.

In [ ]:
# === Ячейка 15: сдвиг порогов решения (добиваем recall у ПРЛ/БАР) ===
import numpy as np
from sklearn.base import clone
from sklearn.model_selection import cross_val_predict, StratifiedKFold

clf = clone(gs.best_estimator_)
classes = np.unique(tr["tag"])              # порядок столбцов decision_function
y_tr = tr["tag"].values

# OOF-оценки решающей функции на train (модель не видела свой тред)
oof = cross_val_predict(clf, tr["doc"], y_tr,
                        cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
                        method="decision_function", n_jobs=N_JOBS)

def macro(scores, y, bias):
    pred = classes[np.argmax(scores + bias, axis=1)]
    return f1_score(y, pred, average="macro")

# координатный спуск по сдвигам (только на OOF train)
bias = np.zeros(len(classes))
grid_b = np.linspace(-3, 3, 61)
for _ in range(3):
    for j in range(len(classes)):
        cur = macro(oof, y_tr, bias)
        best_b = bias[j]
        for b in grid_b:
            t = bias.copy(); t[j] = b
            f = macro(oof, y_tr, t)
            if f > cur:
                cur, best_b = f, b
        bias[j] = best_b
print("CV macro F1: было", round(macro(oof, y_tr, np.zeros(len(classes))), 4),
      "-> стало", round(macro(oof, y_tr, bias), 4))
print("Сдвиги:", dict(zip(classes, np.round(bias, 2))))



## 6. Финальный каскад и оценка

Сначала основная модель выбирает один из шести классов. Затем бинарные модели последовательно уточняют пары ПРЛ/депрессия и БАР/депрессия. Пороги подбираются по OOF на train.

Прогноз обсуждения переносится на все его сообщения. Основной исторический показатель — Macro F1 по сообщениям, поэтому длинные обсуждения сильнее влияют на результат. Отдельно ниже выводится Macro F1 по обсуждениям.

In [ ]:
# === Ячейка 22: финальная архитектура (Stage A + ПРЛ/БАР) + чекпойнт ===
import joblib

# основная модель на всех train-тредах
main = clone(gs.best_estimator_).fit(tr["doc"], tr["tag"])
classes = main.classes_

def fit_binary(minority, major="депрессия"):
    p_tr = tr[tr.tag.isin([minority, major])]
    bp = Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2,
                    max_features=50000, stop_words=RU_STOP)),
                   ("svm", LinearSVC(class_weight="balanced"))])
    oof = cross_val_predict(bp, p_tr["doc"], p_tr["tag"],
                            cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
                            method="decision_function", n_jobs=N_JOBS)
    bp.fit(p_tr["doc"], p_tr["tag"])
    pos = bp.classes_[1]; neg = [c for c in [minority, major] if c != pos][0]
    bt, bf = 0.0, -1
    for t in np.linspace(-2, 2, 81):
        f = f1_score(p_tr["tag"], np.where(oof > t, pos, neg), average="macro")
        if f > bf: bf, bt = f, t
    return dict(model=bp, pos=pos, neg=neg, thr=bt, minority=minority, major=major)

binaries = [fit_binary("ПРЛ"), fit_binary("БАР")]

def classify(docs):                       # Stage A (6 классов + сдвиги) -> каскады ПРЛ, БАР
    out = pd.Series(classes[np.argmax(main.decision_function(docs["doc"]) + bias, axis=1)],
                    index=docs["thread_id"].values)
    for b in binaries:
        sc = pd.Series(b["model"].decision_function(docs["doc"]), index=docs["thread_id"].values)
        verdict = pd.Series(np.where(sc > b["thr"], b["pos"], b["neg"]), index=docs["thread_id"].values)
        m = out.isin([b["minority"], b["major"]])
        out[m] = verdict[m]
    return out

thread_pred_final = classify(te)
row_pred_final = test_df["thread_id"].map(thread_pred_final)
assert row_pred_final.notna().all()
print("Accuracy по сообщениям:", accuracy_score(test_df["tag"], row_pred_final))
print("Macro F1 по обсуждениям:", f1_score(te["tag"], thread_pred_final.reindex(te["thread_id"]), average="macro"))
print("ФИНАЛ TEST macro F1:", round(f1_score(test_df["tag"], row_pred_final, average="macro"), 4))
print(classification_report(test_df["tag"], row_pred_final, digits=3))

joblib.dump({"main": main, "bias": bias, "classes": classes, "binaries": binaries},
            "final_model.joblib")
print("\nСохранено final_model.joblib")

### Сохранённый результат исходного запуска

Источник: последняя ячейка исходного `final_notebook.ipynb`. Это исторический вывод, не результат повторного запуска сокращённого ноутбука.

```text
ФИНАЛ TEST macro F1: 0.8513
                precision    recall  f1-score   support

           БАР      0.895     0.512     0.651       299
           ОКР      0.842     0.753     0.795       969
           ПРЛ      0.958     0.762     0.848       952
     депрессия      0.916     0.955     0.935      9671
тревожное р-во      0.899     0.907     0.903      3164
    шизофрения      1.000     0.952     0.975       418

      accuracy                          0.912     15473
     macro avg      0.918     0.807     0.851     15473
  weighted avg      0.912     0.912     0.910     15473
```

## 7. Пример применения

На вход подаётся полный текст одного обсуждения. Очистка и лемматизация совпадают с обучением. `final_model.joblib` сохраняет модели и пороги; функции предобработки находятся в этом ноутбуке.

In [ ]:
def predict_discussion(text):
    docs = pd.DataFrame({"thread_id": ["example"], "doc": [lemmatize(clean_text(text))]})
    return classify(docs).iloc[0]

# predict_discussion("Полный текст обсуждения...")